### Demo Cross Attention

In [1]:
import torch
import torch.nn.functional as F
import math

# Set random seed for reproducibility
torch.manual_seed(42)

print("--- Step 1: Define dimensions and sequence lengths ---")
batch_size = 1
embed_dim = 4      # Size of the hidden states (d_model)
src_len = 5        # Number of tokens in the encoder source sentence (e.g., "I love machine learning")
tgt_len = 3        # Number of tokens in the decoder target sentence (e.g., "J'aime l'apprentissage")

print(f"Batch size: {batch_size}")
print(f"Embedding Dimension: {embed_dim}")
print(f"Encoder Source Length: {src_len}")
print(f"Decoder Target Length: {tgt_len}\n")


print("--- Step 2: Generate mock Encoder and Decoder hidden states ---")
# Encoder hidden states (Outputs from the last encoder layer)
# Shape: [batch_size, src_len, embed_dim]
encoder_hidden_states = torch.randn(batch_size, src_len, embed_dim)

# Decoder hidden states (Inputs from the current decoder layer)
# Shape: [batch_size, tgt_len, embed_dim]
decoder_hidden_states = torch.randn(batch_size, tgt_len, embed_dim)

print(f"Encoder Hidden States Shape: {encoder_hidden_states.shape}")
print(f"Decoder Hidden States Shape: {decoder_hidden_states.shape}\n")


print("--- Step 3: Project to Queries, Keys, and Values ---")
# Create linear projection layers
W_q = torch.nn.Linear(embed_dim, embed_dim, bias=False)
W_k = torch.nn.Linear(embed_dim, embed_dim, bias=False)
W_v = torch.nn.Linear(embed_dim, embed_dim, bias=False)

# CRITICAL STEP: Queries come from Decoder, Keys/Values come from Encoder
Q = W_q(decoder_hidden_states)     # Shape: [batch_size, tgt_len, embed_dim]
K = W_k(encoder_hidden_states)     # Shape: [batch_size, src_len, embed_dim]
V = W_v(encoder_hidden_states)     # Shape: [batch_size, src_len, embed_dim]

print(f"Queries (Q) Shape [from Decoder]: {Q.shape}")
print(f"Keys (K) Shape [from Encoder]: {K.shape}")
print(f"Values (V) Shape [from Encoder]: {V.shape}\n")


print("--- Step 4: Calculate Raw Attention Scores ---")
# Transpose K for matrix multiplication: [batch_size, embed_dim, src_len]
K_t = K.transpose(-2, -1)

# Compute dot product between Q and K
# [batch_size, tgt_len, embed_dim] x [batch_size, embed_dim, src_len] -> [batch_size, tgt_len, src_len]
scores = torch.matmul(Q, K_t)
print(f"Raw Scores Shape: {scores.shape}")
print("Raw Scores Vector:\n", scores, "\n")


print("--- Step 5: Scale Scores to prevent exploding gradients ---")
# Scale by the square root of the embedding dimension
d_k = embed_dim
scaled_scores = scores / math.sqrt(d_k)
print("Scaled Scores Vector:\n", scaled_scores, "\n")


print("--- Step 6: Apply Softmax to get Attention Weights ---")
# Softmax across the source sequence length dimension (dim=-1)
# This maps out exactly how much each decoder token attends to each encoder token
attention_weights = F.softmax(scaled_scores, dim=-1)

print(f"Attention Weights Shape: {attention_weights.shape}")
print("Attention Weights (Rows sum to 1.0):\n", attention_weights)
print("Row Verification Sums:", attention_weights.sum(dim=-1), "\n")


print("--- Step 7: Compute the Final Attention Output ---")
# Multiply attention weights by the Values matrix
# [batch_size, tgt_len, src_len] x [batch_size, src_len, embed_dim] -> [batch_size, tgt_len, embed_dim]
output = torch.matmul(attention_weights, V)

print(f"Final Cross-Attention Output Shape: {output.shape}")
print("Output Vector:\n", output)


--- Step 1: Define dimensions and sequence lengths ---
Batch size: 1
Embedding Dimension: 4
Encoder Source Length: 5
Decoder Target Length: 3

--- Step 2: Generate mock Encoder and Decoder hidden states ---
Encoder Hidden States Shape: torch.Size([1, 5, 4])
Decoder Hidden States Shape: torch.Size([1, 3, 4])

--- Step 3: Project to Queries, Keys, and Values ---
Queries (Q) Shape [from Decoder]: torch.Size([1, 3, 4])
Keys (K) Shape [from Encoder]: torch.Size([1, 5, 4])
Values (V) Shape [from Encoder]: torch.Size([1, 5, 4])

--- Step 4: Calculate Raw Attention Scores ---
Raw Scores Shape: torch.Size([1, 3, 5])
Raw Scores Vector:
 tensor([[[ 1.1893, -0.8154,  0.1319, -0.4277,  0.4639],
         [ 0.2016, -0.4598,  0.2421, -0.0445, -0.1151],
         [ 0.0135,  0.0742, -0.0431,  0.0111, -0.0173]]],
       grad_fn=<UnsafeViewBackward0>) 

--- Step 5: Scale Scores to prevent exploding gradients ---
Scaled Scores Vector:
 tensor([[[ 0.5946, -0.4077,  0.0660, -0.2138,  0.2319],
         [ 0.100